# Stream Ciphers

## Introduction

Stream ciphers are encryption algorithms that encrypt data bit-by-bit or byte-by-byte, instead of in blocks.
They were first widely used in the 1940s with devices like the German Enigma machine and later one-time pads.
Modern stream ciphers aim to be fast, lightweight, and secure for real-time communications (e.g., mobile phones, IoT devices).

Methods used here:

- LFSR-based pseudorandom generation

- Alternating Step Generator for better security

- Cryptanalysis with Berlekamp-Massey and Known-Plaintext Attack
  
However, stream ciphers can be vulnerable if the same keystream is reused (key reuse attacks), if the initial state is predictable, or if enough keystream bits are leaked to allow cryptanalysis methods like the Berlekamp-Massey algorithm to recover the internal structure.

Today, stream ciphers are commonly used in wireless networks (like Bluetooth, GSM) and low-power embedded systems.

In [ ]:
from bits import Bits
from bitgenerator import AlternatingStep
from lfsr import LFSR, berlekamp_massey

### Bits Class

The `Bits` class represents and manipulates sequences of bits.

**Methods:**
- Initialization from list, int, or bytes
- XOR, AND, addition, and replication operations
- Conversion to bytes
- Parity bit calculation

**Example:**

In [ ]:
# Example usage:
if __name__ == "__main__":
    b1 = Bits([True, False, True, True])  
    print(b1)  # 1011
    print(b1.parity_bit())  # 1 (odd parity)
    
    b2 = Bits(0x6a, length=8)  
    print(b2)  # 01101010
    
    b3 = Bits(b'a')  
    print(b3)  # 01100001
    
    b4 = b1 + b2  # Concatenate b1 and b2
    print(b4)  # 101101101010
    
    b5 = b2 * 2  # Replicate b2 by 2
    print(b5)  # 0110101001101010
    
    b6 = b1 ^ b2  # XOR operation
    print(b6)  # 11011010 (XOR result)

# 1. LFSR

An LFSR is a simple shift register where the input bit is a linear function (XOR) of selected bits from the
current state (called taps).
First used for error detection and pseudorandom generation (e.g., satellite communications, early cryptography).

Key point:

- They generate sequences that look random but are completely determined by the initial state.

Today, LFSRs are still used in lightweight cryptography, secure codes, and random number generators where high speed is needed.

### LFSR Class

The `LFSR` class implements a Linear Feedback Shift Register.

**Methods:**
- Initialization with taps and optional state
- Bit generation with feedback
- Full cycle generation
- Step execution

**Example:**

In [ ]:
# Define the feedback polynomial for P(x) = x^12 + x^6 + x^4 + x + 1
# Taps are at positions 12, 6, 4, 1, 0
poly = {12, 6, 4, 1, 0}

# Define the initial state as a Bits object
# The state 0b110101101011 is equivalent to the binary string '110101101011'
init_state = ([1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1])  # Using Bits class to initialize state

# Create the LFSR instance
lfsr = LFSR(poly=poly, state=init_state)

# Show the initial configuration
print("Initial LFSR configuration:")
print(lfsr)
print()

# Generate the first 8 output bits manually
print("First 8 output bits (using next()):")
for _ in range(8):
    bit = next(lfsr)
    print(int(bit), end=' ')
print('\n')

# Run 10 steps and print the bits
print("Next 10 output bits (using run_steps()):")
bits = lfsr.run_steps(N=10)
print(bits)
print()

# Now, run a full cycle and display it
print("Running a full cycle (using cycle()):")
full_cycle = lfsr.cycle()
print(full_cycle)
print()

# Check cycle length
print(f"Cycle length: {len(full_cycle)} bits")


## 2. Berlekamp-Massey Algorithm

The Berlekamp-Massey algorithm was invented in the 1960s by Elwyn Berlekamp and James Massey.
It finds the shortest LFSR that can reproduce a given binary sequence.

Usage:

- In cryptography, to break stream ciphers (if part of the keystream is known).

- In communications, for error correction (like decoding BCH and Reed-Solomon codes).

Today, it is a key tool in both cryptographic attacks and error-correcting code designs.

**Example:**

In [ ]:
# Test Berlekamp-Massey using the binary_sequence.bin
if __name__ == "__main__":
    with open("binary_sequence.bin", "rb") as f:
        data = f.read()
        bits = Bits(data)

    poly, complexity = berlekamp_massey(bits)
    print("Feedback Polynomial (as degrees):", poly)
    print("Linear Complexity:", complexity)

## 3. Alternating-Step Generator

The Alternating Step Generator is a cryptographic technique that uses multiple LFSRs and irregular clocking (steps decided by a control LFSR) to make the output harder to predict.

Key idea:

- Instead of regular stepping, we use one LFSR to control others.

- This adds complexity and makes classical attacks much harder.

Alternating step generators are a building block in some modern secure stream cipher designs and lightweight encryption protocols.



In [ ]:
# Example polynomials and seed
polyC = [0, 2, 5]  # Example polynomial for lfsrC
poly0 = [0, 1, 3]  # Example polynomial for lfsr0
poly1 = [0, 1, 4]  # Example polynomial for lfsr1
seed = [1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1]  # 15 bits

# Instantiate the generator
gen = AlternatingStep(seed=seed, polyC=polyC, poly0=poly0, poly1=poly1)

# Generate or inspect step by step
print("Ctrl, Out0, Out1, Keystream")
for _ in range(10):
    # Generate the next keystream bit
    keystream_bit = next(gen)
    
    # Access the outputs
    ctrl = int(gen.lfsrC.output)   # Feedback from lfsrC
    d0   = int(gen.lfsr0.output)   # Output from lfsr0
    d1   = int(gen.lfsr1.output)   # Output from lfsr1
    print(ctrl, d0, d1, keystream_bit)



### Result: 
The Alternating Step Generator was successfully implemented using three LFSRs with the following polynomials:

- LFSR-C: [0, 2, 5]

- LFSR-0: [0, 1, 3]

- LFSR-1: [0, 1, 4]

The generator produced a keystream by alternating between the LFSR outputs. The results showed an alternating pattern, with the keystream bit oscillating between True and False. This demonstrates the generator's functionality in producing pseudo-random keystreams for cryptographic applications.

## 4. Known-Plaintext Attack (KPA)

A Known-Plaintext Attack (KPA) happens when the attacker knows some pieces of the original plaintext and the corresponding ciphertext.
This can be exploited to recover the secret keystream and reconstruct the cipher structure (LFSR taps and state).

How we use it:

- XOR known plaintext and ciphertext to recover partial keystream.

- Use Berlekamp-Massey to find the feedback polynomial.

- Rebuild the LFSR and decrypt the full message.

KPA is still a relevant risk today in systems that don't refresh keys often or leak small amounts of plaintext (e.g., predictable headers).

**Example:**

In [ ]:
def kpa_attack(ciphertext_bits, known_plaintext_bits):
    """
    Perform a Known Plaintext Attack to decrypt the full ciphertext.
    
    Args:
        ciphertext_bits (Bits): The full ciphertext bits from ciphertext.bin
        known_plaintext_bits (Bits): The known initial plaintext bits from known-plaintext.txt

    Returns:
        tuple(str, list): (Recovered plaintext as UTF-8 string, Recovered feedback polynomial taps)
    """
    # Step 1: Recover part of the keystream
    known_ciphertext_part = Bits(ciphertext_bits[:len(known_plaintext_bits)])
    recovered_keystream = known_ciphertext_part ^ known_plaintext_bits

    # Step 2: Run Berlekamp-Massey on the recovered keystream
    feedback_poly, lfsr_length = berlekamp_massey(recovered_keystream.bits)

    if lfsr_length == 0:
        raise ValueError("Failed to recover LFSR structure. Not enough known plaintext.")

    # Step 3: Recover initial state (first lfsr_length bits of recovered keystream)
    initial_state = recovered_keystream.bits[:lfsr_length]

    # Step 4: Build the recovered LFSR
    lfsr = LFSR(feedback_poly, state=initial_state)

    # Step 5: Generate full keystream
    full_keystream = lfsr.run_steps(len(ciphertext_bits))

    # Step 6: Decrypt the full ciphertext
    recovered_plaintext_bits = ciphertext_bits ^ full_keystream

    # Step 7: Convert recovered plaintext bits into bytes and decode as UTF-8
    recovered_plaintext_bytes = recovered_plaintext_bits.to_bytes()
    recovered_plaintext = recovered_plaintext_bytes.decode('utf-8', errors='replace')

    return recovered_plaintext, feedback_poly

# 3. Load files
with open('known-plaintext.txt', 'rb') as f:
    known_plaintext_bytes = f.read()
known_plaintext_bits = Bits(known_plaintext_bytes)

with open('ciphertext.bin', 'rb') as f:
    ciphertext_bytes = f.read()
ciphertext_bits = Bits(ciphertext_bytes)

# 4. Run the KPA
plaintext, poly = kpa_attack(ciphertext_bits, known_plaintext_bits)

# 5. Print results
print("\n=== Recovered Full Plaintext ===")
print(plaintext)

print("\n=== Recovered Feedback Polynomial (taps) ===")
print(poly)

## Summary

This project focused on analyzing and attacking a stream cipher using LFSRs (Linear Feedback Shift Registers). It included the following steps:

- LFSR Implementation: A custom LFSR generator was created using example polynomials like [0, 2, 5] for LFSR-C, [0, 1, 3] for LFSR-0, and [0, 1, 4] for LFSR-1. The generator alternates between LFSRs and produces a keystream, which was tested step-by-step.

- Alternating Step Generator: The alternating generator effectively combined outputs from three different LFSRs. For example, the feedback polynomial for LFSR-C was [0, 2, 5], and the alternating generator used both LFSR-0 and LFSR-1 outputs to produce a mixed keystream. The generator was tested for 10 steps, and the results showed how it combines different LFSRs.

- Berlekamp-Massey Algorithm: The algorithm was applied to the keystream, resulting in the feedback polynomial [0, 7, 18], along with a linear complexity of 18. This step successfully identified the structure of the stream cipher.

- Known Plaintext Attack (KPA): A known plaintext attack was attempted, and although part of the ciphertext was decrypted, full recovery of the plaintext was not possible due to the limited length of the known plaintext.

## Conclusion:
The project demonstrated successful implementation of LFSRs, an alternating step generator, and the Berlekamp-Massey algorithm. The KPA showed partial success, but full decryption was hindered by the limited known plaintext.